In [4]:
import os
import yfinance as yf
import time
import smtplib
from email.message import EmailMessage
import pandas as pd 
STOP_LOSS_PCT = -0.01 
TRAILING_STOP_PCT = -0.015  

def log_system_event(message):
    timestamp = time.ctime()
    with open('system_log.txt', 'a') as f:
        f.write(f'[{timestamp}], {message}\n')
    print(f'logged {message}')
    
def send_trade_notification(signal, price, pnl):
    msg = EmailMessage()
    msg.set_content(f'Update From The Bot:\nAction: {signal}\nPrice: {price:.2f}\nPnL: {pnl:.2f}')
    msg['Subject'] = f'Bot Alert: {signal}'
    msg['From'] = 'smtp.quant@gmail.com'
    msg['To'] = 'smtp.quant@gmail.com'
    
    with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
        smtp.login('smtp.quant@gmail.com', 'bjxq rsbe ocmn fobw')
        smtp.send_message(msg)


last_buy_price = 0
last_signal = 'None'
highest_price = 0

if os.path.exists('trade_log.csv'):
    try:
        with open('trade_log.csv','r') as f:
            lines = f.readlines()
            if len(lines) > 1:
                last_line = lines[-1].split(',')
                last_signal = last_line[2].strip()
                if last_signal == 'BUY':
                    last_buy_price = float(last_line[1])
                    highest_price = last_buy_price # Initialize peak at buy price if recovered
        print(f' Memory recovered. Last signal was: {last_signal}')
    except Exception as e:
        print(f' Memory recovery failed: {e}. Starting fresh.')
print(' Quantum Engine: Online (Trailing Stop-Loss Active)')

while True:
        
    data = None
    while data is None:
        try:
            data = yf.download('RELIANCE.NS', period='5d', interval='1m', progress=False)
            if data.empty:
                data = None
                raise ValueError('data is empty')
        except Exception as e:
            print(f' Internet Hipcup {e}, Retrying in 30 sec...')
            time.sleep(30)
    
            data['SMA_45'] = data['Close']['RELIANCE.NS'].rolling(window=45).mean()
            data['SMA_190'] = data['Close']['RELIANCE.NS'].rolling(window=190).mean()
            
            close_prices = data['Close']['RELIANCE.NS']
            data['Delta'] = close_prices.diff()
            data['Gain'] = data['Delta'].clip(lower=0)
            data['Loss'] = data['Delta'].clip(upper=0).abs()
            data['Avg_Gain'] = data['Gain'].rolling(window=14).mean()
            data['Avg_Loss'] = data['Loss'].rolling(window=14).mean()
            data['RS'] = data['Avg_Gain'] / data['Avg_Loss']
            data['RSI'] = 100 - (100 / (1 + data['RS']))
            
            latest = data.iloc[-1]
            sma_45_val = latest['SMA_45'].item()
            sma_190_val = latest['SMA_190'].item()
            current_price = latest['Close']['RELIANCE.NS'].item()
            current_rsi = latest['RSI'].item()
            
            if last_signal == 'BUY' and last_buy_price > 0:
                current_pnl_pct = (current_price - last_buy_price) / last_buy_price 
                if highest_price == 0 or current_price > highest_price:
                    highest_price = current_price
                drawdown_from_peak = (current_price - highest_price) / highest_price
                trigger_signal = None
                if current_pnl_pct <= STOP_LOSS_PCT:
                    trigger_signal = 'STOP_LOSS'
                elif drawdown_from_peak <= TRAILING_STOP_PCT:
                    trigger_signal = 'TRAILING_STOP' 
                if trigger_signal:
                    pnl = current_price - last_buy_price
                    print(f'\n {trigger_signal} HIT! Force Selling at {current_price:.2f} | PnL: {current_pnl_pct:.2%}')
                    with open('trade_log.csv', 'a') as f:
                        f.write(f'{time.ctime()},{current_price},{trigger_signal},{pnl}\n')
                    try:
                        send_trade_notification(trigger_signal, current_price, pnl)
                        print(' Dynamic Exit Notification Sent')
                    except Exception as e:
                        print(f' Notification Failed: {e}')
                    last_signal = 'COOLDOWN'
                    last_buy_price = 0
                    highest_price = 0 
                    time.sleep(60)
                    continue 
            if sma_45_val > sma_190_val and current_rsi < 60:
                current_signal = 'BUY'
            else:
                current_signal = 'SELL'
    
            if last_signal == 'COOLDOWN':
                if current_signal == 'SELL':
                    print('\n Cooldown finished. Market trend shifted down. System reset.')
                    last_signal = 'SELL'
                else:
                    pnl_display = 0
                    print(f'{time.ctime()} | {current_price:.2f} | Status: COOLDOWN (Waiting for trend reset)')
                    
            elif current_signal != last_signal:
                pnl = 0
                if current_signal == 'SELL' and last_buy_price > 0:
                    pnl = current_price - last_buy_price
                    print(f'\n Trade Closed: PnL {pnl:.2f}')
                    last_buy_price = 0 
                    highest_price = 0
                if current_signal == 'BUY':
                    last_buy_price = current_price
                    highest_price = current_price 
                    print(f'\n Entry: Buying at {current_price:.2f} (RSI: {current_rsi:.2f})')
                with open('trade_log.csv','a') as f:
                    f.write(f'{time.ctime()},{current_price},{current_signal},{pnl}\n')
                    
                try:
                    send_trade_notification(current_signal, current_price, pnl)
                    print(' Notification Sent')
                except Exception as e:
                    print(f' Notification Failed: {e}')
                last_signal = current_signal
            else:
                pnl_display = current_price - last_buy_price if last_signal == 'BUY' else 0
                rsi_display = f"{current_rsi:.2f}" if not pd.isna(current_rsi) else "N/A"
                if last_signal == 'BUY':
                    trail_display = f" | Peak: {highest_price:.2f} | Drawdown: {(current_price - highest_price)/highest_price:.2%}"
                else:
                    trail_display = ""
                print(f'{time.ctime()} | Price: {current_price:.2f} | RSI: {rsi_display} | Signal: {current_signal} | Open PnL: {pnl_display:.2f}{trail_display}')
        except Exception as e:
            print(f'\n⚠️ System Alert: {e}')
            error_msg = f'Critical Error:{e}'
            print(error_msg)
        try:
            send_trade_notification('System Alert',0,0)
        except:
            pass
            print('Restarting loop in 15 sec...')
            time.sleep(15)
            continue  
        time.sleep(60)

 Memory recovered. Last signal was: BUY
 Quantum Engine: Online (Trailing Stop-Loss Active)
Restarting loop in 15 sec...
Restarting loop in 15 sec...
Restarting loop in 15 sec...
Restarting loop in 15 sec...


KeyboardInterrupt: 

In [ ]:
import os
import yfinance as yf
import time
import smtplib
from email.message import EmailMessage
import pandas as pd 

STOP_LOSS_PCT = -0.01 
TRAILING_STOP_PCT = -0.015  
def log_system_event(message):
    timestamp = time.ctime()
    with open('system_log.txt', 'a') as f:
        f.write(f'[{timestamp}], {message}\n')
    print(f'Logged: {message}')
    
def send_trade_notification(signal, price, pnl):
    msg = EmailMessage()
    msg.set_content(f'Update From The Bot:\nAction: {signal}\nPrice: {price:.2f}\nPnL: {pnl:.2f}')
    msg['Subject'] = f'Bot Alert: {signal}'
    msg['From'] = 'smtp.quant@gmail.com'
    msg['To'] = 'smtp.quant@gmail.com'
    
    with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
        smtp.login('smtp.quant@gmail.com', 'My pass')
        smtp.send_message(msg)
last_buy_price = 0
last_signal = 'None'
highest_price = 0

if os.path.exists('trade_log.csv'):
    try:
        with open('trade_log.csv','r') as f:
            lines = f.readlines()
            if len(lines) > 1:
                last_line = lines[-1].split(',')
                last_signal = last_line[2].strip()
                if last_signal == 'BUY':
                    last_buy_price = float(last_line[1])
                    highest_price = last_buy_price
        print(f' Memory recovered. Last signal: {last_signal}')
    except Exception as e:
        print(f' Memory recovery failed: {e}')

print(' Quantum Engine: Online')
while True:
    try:
        data = None
        while data is None:
            try:
                data = yf.download('RELIANCE.NS', period='5d', interval='1m', progress=False)
                if data.empty:
                    data = None
                    raise ValueError('Data is empty')
            except Exception as e:
                log_system_event(f"Internet Hiccup: {e}. Retrying in 30s...")
                time.sleep(30)
        data['SMA_45'] = data['Close']['RELIANCE.NS'].rolling(window=45).mean()
        data['SMA_190'] = data['Close']['RELIANCE.NS'].rolling(window=190).mean()
        close_prices = data['Close']['RELIANCE.NS']
        data['Delta'] = close_prices.diff()
        data['Gain'] = data['Delta'].clip(lower=0)
        data['Loss'] = data['Delta'].clip(upper=0).abs()
        data['Avg_Gain'] = data['Gain'].rolling(window=14).mean()
        data['Avg_Loss'] = data['Loss'].rolling(window=14).mean()
        data['RS'] = data['Avg_Gain'] / data['Avg_Loss']
        data['RSI'] = 100 - (100 / (1 + data['RS']))
        latest = data.iloc[-1]
        sma_45_val = latest['SMA_45'].item()
        sma_190_val = latest['SMA_190'].item()
        current_price = latest['Close']['RELIANCE.NS'].item()
        current_rsi = latest['RSI'].item()
        if last_signal == 'BUY' and last_buy_price > 0:
            current_pnl_pct = (current_price - last_buy_price) / last_buy_price 
            if highest_price == 0 or current_price > highest_price:
                highest_price = current_price
            drawdown_from_peak = (current_price - highest_price) / highest_price
            trigger_signal = None
            if current_pnl_pct <= STOP_LOSS_PCT:
                trigger_signal = 'STOP_LOSS'
            elif drawdown_from_peak <= TRAILING_STOP_PCT:
                trigger_signal = 'TRAILING_STOP' 
            if trigger_signal:
                pnl = current_price - last_buy_price
                print(f'\n {trigger_signal} HIT! Selling at {current_price:.2f}')
                with open('trade_log.csv', 'a') as f:
                    f.write(f'{time.ctime()},{current_price},{trigger_signal},{pnl}\n')
                try:
                    send_trade_notification(trigger_signal, current_price, pnl)
                except Exception as e:
                    print(f'Notification Failed: {e}')
                last_signal = 'COOLDOWN'
                last_buy_price = 0
                highest_price = 0 
                time.sleep(60)
                continue 
        if sma_45_val > sma_190_val and current_rsi < 60:
            current_signal = 'BUY'
        else:
            current_signal = 'SELL'
        if last_signal == 'COOLDOWN':
            if current_signal == 'SELL':
                print('\n Cooldown finished. System reset.')
                last_signal = 'SELL'
            else:
                print(f'{time.ctime()} | Status: COOLDOWN')
        elif current_signal != last_signal:
            pnl = 0
            if current_signal == 'SELL' and last_buy_price > 0:
                pnl = current_price - last_buy_price
                last_buy_price = 0 
                highest_price = 0
            if current_signal == 'BUY':
                last_buy_price = current_price
                highest_price = current_price 
            with open('trade_log.csv','a') as f:
                f.write(f'{time.ctime()},{current_price},{current_signal},{pnl}\n')
            send_trade_notification(current_signal, current_price, pnl)
            last_signal = current_signal
        else:
            pnl_display = current_price - last_buy_price if last_signal == 'BUY' else 0
            rsi_display = f"{current_rsi:.2f}" if not pd.isna(current_rsi) else "N/A"
            trail_display = f" | Peak: {highest_price:.2f}" if last_signal == 'BUY' else ""
            print(f'{time.ctime()} | Price: {current_price:.2f} | RSI: {rsi_display} | Signal: {last_signal}{trail_display}')
    except Exception as e:
        error_msg = f'CRITICAL ERROR: {e}'
        print(f'\n {error_msg}')
        log_system_event(error_msg)
        try:
            send_trade_notification('System Alert', 0, 0)
        except:
            pass
        print('Restarting loop in 15 sec...')
        time.sleep(15)
        continue  
    time.sleep(60)

🤖 Memory recovered. Last signal: STOP_LOSS
🚀 Quantum Engine: Online
Mon May 11 16:14:15 2026 | Price: 1385.00 | RSI: 30.00 | Signal: SELL
Mon May 11 16:15:15 2026 | Price: 1385.00 | RSI: 30.00 | Signal: SELL
Mon May 11 16:16:15 2026 | Price: 1385.00 | RSI: 30.00 | Signal: SELL
Mon May 11 16:17:16 2026 | Price: 1385.00 | RSI: 30.00 | Signal: SELL
Mon May 11 16:18:16 2026 | Price: 1385.00 | RSI: 30.00 | Signal: SELL
Mon May 11 16:19:17 2026 | Price: 1385.00 | RSI: 30.00 | Signal: SELL
